In [1]:
import duckdb
import pandas as pd

In [2]:
df_zones = pd.read_csv("../data/data_zone_urb/raw/zones_urbaines.csv")
df_zones

,SIREN,Nom de l'EPCI,Nature de l'EPCI,Superficie de l'EPCI (km²),Superficie des territoires artificialisés* (km²),Part de la superficie artificialisée,Unnamed: 6,* Les donnes proviennent de Corine Land Cover millésime 2018
0,200000172,CC Faucigny-Glières,Communauté de communes,"151,207","15,745","10,41 %",NaN,NaN
1,200000438,CC du Pays de Pontchâteau Saint-Gildas-des-Bois,Communauté de communes,"327,68","32,093","9,79 %",NaN,NaN
2,200000545,CC des Portes de Romilly-sur-Seine,Communauté de communes,"104,975","12,727","12,12 %",NaN,NaN
3,200000628,CC Rhône Lez Provence,Communauté de communes,"150,602","14,459","9,60 %",NaN,NaN
4,200000800,CC Cœur de Sologne,Communauté de communes,"343,024","10,642","3,10 %",NaN,NaN
...,...,...,...,...,...,...,...,...
1238,242020071,CC du Centre Corse,Communauté de communes,"360,435","4,71","1,31 %",NaN,NaN
1239,200042943,CC du Cap Corse,Communauté de communes,"307,328","6,086","1,98 %",NaN,NaN
1240,242010056,CA du Pays Ajaccien,Communauté d'agglomération,"270,03","33,014","12,23 %",NaN,NaN
1241,242000354,CA de Bastia,Communauté d'agglomération,"69,221","16,002","23,12 %",NaN,NaN


In [3]:
df_zones.drop('Unnamed: 6', axis=1, inplace=True)

In [4]:
df_zones.drop('* Les donnes proviennent de Corine Land Cover millésime 2018', axis=1, inplace=True)

In [5]:
mapping = {'SIREN':'siren', "Nom de l'EPCI":'nom_epci', "Nature de l'EPCI":'nature_epci',
       "Superficie de l'EPCI (km²)":"superficie_epci",
       'Superficie des territoires artificialisés* (km²)':'superficie_artificialisee',
       'Part de la superficie artificialisée':'part_percent_superficie_artificialisee'}

df_zones.rename(columns=mapping, inplace=True)

In [6]:
df_zones["superficie_epci"] = df_zones["superficie_epci"].replace(',', '.', regex=True).astype(float)
df_zones["superficie_artificialisee"] = df_zones["superficie_artificialisee"].replace(',', '.', regex=True).astype(float)
df_zones["part_percent_superficie_artificialisee"] = df_zones["part_percent_superficie_artificialisee"].replace(',', '.', regex=True).replace(' %', '', regex=True).astype(float)

In [7]:
df_zones

,siren,nom_epci,nature_epci,superficie_epci,superficie_artificialisee,part_percent_superficie_artificialisee
0,200000172,CC Faucigny-Glières,Communauté de communes,151.207,15.745,10.41
1,200000438,CC du Pays de Pontchâteau Saint-Gildas-des-Bois,Communauté de communes,327.680,32.093,9.79
2,200000545,CC des Portes de Romilly-sur-Seine,Communauté de communes,104.975,12.727,12.12
3,200000628,CC Rhône Lez Provence,Communauté de communes,150.602,14.459,9.60
4,200000800,CC Cœur de Sologne,Communauté de communes,343.024,10.642,3.10
...,...,...,...,...,...,...
1238,242020071,CC du Centre Corse,Communauté de communes,360.435,4.710,1.31
1239,200042943,CC du Cap Corse,Communauté de communes,307.328,6.086,1.98
1240,242010056,CA du Pays Ajaccien,Communauté d'agglomération,270.030,33.014,12.23
1241,242000354,CA de Bastia,Communauté d'agglomération,69.221,16.002,23.12


In [8]:
df_epci = pd.read_csv("../data/processed/epci_membres.csv")
df_epci

,code_insee,nom,pop_tot_commune,pop_mun_commune,siren,epci_nom,epci_type,epci_modeFinancement,total_pop_tot,total_pop_mun,superficie_hectare,superficie_km2,dept_com,bassin_vie,dept_epci
0,01304,Pont-d'Ain,2912,2862,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,FPU,15156,14873,1122.0,11.0,01,01304,01
1,01199,Jujurieux,2246,2209,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,FPU,15156,14873,1548.0,15.0,01,01004,01
2,01363,Saint-Jean-le-Vieux,1873,1799,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,FPU,15156,14873,1517.0,15.0,01,01004,01
3,01314,Priay,1826,1803,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,FPU,15156,14873,1571.0,16.0,01,01004,01
4,01273,Neuville-sur-Ain,1823,1798,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,FPU,15156,14873,1991.0,20.0,01,01004,01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34996,97601,Acoua,5384,5192,200060465,CA du Grand Nord de Mayotte,CA,FPU,60372,59042,1290.0,13.0,97,97611,976
34997,97603,Bandrele,10529,10282,200060473,CC du Sud,CC,FPU,31945,30898,3514.0,35.0,97,97611,976
34998,97606,Chirongui,9197,8920,200060473,CC du Sud,CC,FPU,31945,30898,2590.0,26.0,97,97611,976
34999,97604,Bouéni,6503,6189,200060473,CC du Sud,CC,FPU,31945,30898,1381.0,14.0,97,97611,976


In [9]:
query = """
SELECT
    epci.siren AS siren,
    epci.superficie_km2,
    z.superficie_artificialisee,
    z.part_percent_superficie_artificialisee
FROM df_epci epci
LEFT JOIN df_zones z
ON z.siren = epci.siren
"""

df_zone_urbanise_final = duckdb.sql(query)


In [10]:
df_zone_urbanise_final

┌───────────┬────────────────┬───────────────────────────┬────────────────────────────────────────┐
│   siren   │ superficie_km2 │ superficie_artificialisee │ part_percent_superficie_artificialisee │
│   int64   │     double     │          double           │                 double                 │
├───────────┼────────────────┼───────────────────────────┼────────────────────────────────────────┤
│ 200029999 │           11.0 │                    11.589 │                                   6.78 │
│ 200029999 │           15.0 │                    11.589 │                                   6.78 │
│ 200029999 │           15.0 │                    11.589 │                                   6.78 │
│ 200029999 │           16.0 │                    11.589 │                                   6.78 │
│ 200029999 │           20.0 │                    11.589 │                                   6.78 │
│ 200029999 │           20.0 │                    11.589 │                                   6.78 │


In [37]:
df_amenagement_cyclable = duckdb.read_parquet("../data/data_zone_urb/raw/amenagement_cyclable.parquet")
df_amenagement_cyclable

┌──────────────────────┬────────────┬─────────┬────────────┬────────────────┬──────────────────┬─────────────────┬───────────┬──────────┬────────────┬─────────┬────────────┬────────────────┬──────────────────┬─────────────────┬───────────┬──────────┬────────────┬─────────┬────────────┬───────────┬────────────┬─────────┬───────────┬──────────────────────┬───────────┬───────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [38]:
import geopandas as gpd
from shapely import wkb

# 1. Charger l'extension sur l'instance par défaut de DuckDB
duckdb.sql("INSTALL spatial;")
duckdb.sql("LOAD spatial;")

query = """
    SELECT * EXCLUDE (geometry), 
           ST_AsWKB(geometry) AS geometry 
    FROM df_amenagement_cyclable
"""

df_amenagement_cyclable = duckdb.sql(query).df()
df_amenagement_cyclable


,id_local,id_osm,num_iti,code_com_d,ame_d,regime_d,sens_d,largeur_d,local_d,statut_d,...,revet_g,access_ame,date_maj,trafic_vit,lumiere,d_service,source,project_c,ref_geo,geometry
0,geovelo_1426115274_2B033,1426115274,None,2B033,AUCUN,EN AGGLOMERATION,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,LISSE,None,NaT,50.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 4, 0, 0, 0, 108, 7, 35, 246, 9..."
1,geovelo_621800068_2B007,621800068,None,2B007,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,None,VTC,NaT,5.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 2, 0, 0, 0, 191, 241, 181, 103..."
2,geovelo_363780647_66148,363780647,,66148,AUTRE,None,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,None,None,NaT,NaN,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 2, 0, 0, 0, 66, 249, 113, 234,..."
3,geovelo_187713892_66093,187713892,None,66001,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,None,VTC,NaT,5.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 6, 0, 0, 0, 77, 246, 207, 211,..."
4,geovelo_187713892_66093,187713892,None,66093,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,None,VTC,NaT,5.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 38, 2, 0, 0, 196, 119, 27, 187..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
381851,geovelo_150112693_65192,150112693,None,65192,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,RUGUEUX,VTC,NaT,5.0,0.0,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 10, 0, 0, 0, 16, 110, 124, 55,..."
381852,geovelo_128925830_65192,128925830,None,65192,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,RUGUEUX,VTC,NaT,5.0,0.0,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 32, 0, 0, 0, 65, 154, 86, 175,..."
381853,geovelo_267690975_65192,267690975,None,65192,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,RUGUEUX,VTC,NaT,5.0,0.0,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 6, 0, 0, 0, 180, 177, 109, 172..."
381854,geovelo_698615538_65192,698615538,None,65192,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,RUGUEUX,VTC,NaT,5.0,0.0,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 2, 0, 0, 0, 175, 127, 50, 33, ..."


In [17]:
df_temp = df_amenagement_cyclable.dropna(subset=['code_com_d'])

In [18]:
df_temp

,id_local,id_osm,num_iti,code_com_d,ame_d,regime_d,sens_d,largeur_d,local_d,statut_d,...,revet_g,access_ame,date_maj,trafic_vit,lumiere,d_service,source,project_c,ref_geo,geometry
0,geovelo_1426115274_2B033,1426115274,None,2B033,AUCUN,EN AGGLOMERATION,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,LISSE,None,NaT,50.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 4, 0, 0, 0, 108, 7, 35, 246, 9..."
1,geovelo_621800068_2B007,621800068,None,2B007,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,None,VTC,NaT,5.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 2, 0, 0, 0, 191, 241, 181, 103..."
2,geovelo_363780647_66148,363780647,,66148,AUTRE,None,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,None,None,NaT,NaN,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 2, 0, 0, 0, 66, 249, 113, 234,..."
3,geovelo_187713892_66093,187713892,None,66001,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,None,VTC,NaT,5.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 6, 0, 0, 0, 77, 246, 207, 211,..."
4,geovelo_187713892_66093,187713892,None,66093,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,None,VTC,NaT,5.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 38, 2, 0, 0, 196, 119, 27, 187..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
381851,geovelo_150112693_65192,150112693,None,65192,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,RUGUEUX,VTC,NaT,5.0,0.0,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 10, 0, 0, 0, 16, 110, 124, 55,..."
381852,geovelo_128925830_65192,128925830,None,65192,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,RUGUEUX,VTC,NaT,5.0,0.0,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 32, 0, 0, 0, 65, 154, 86, 175,..."
381853,geovelo_267690975_65192,267690975,None,65192,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,RUGUEUX,VTC,NaT,5.0,0.0,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 6, 0, 0, 0, 180, 177, 109, 172..."
381854,geovelo_698615538_65192,698615538,None,65192,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,RUGUEUX,VTC,NaT,5.0,0.0,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 2, 0, 0, 0, 175, 127, 50, 33, ..."


In [19]:
df_temp[df_temp["code_com_d"].str.startswith("132")]

,id_local,id_osm,num_iti,code_com_d,ame_d,regime_d,sens_d,largeur_d,local_d,statut_d,...,revet_g,access_ame,date_maj,trafic_vit,lumiere,d_service,source,project_c,ref_geo,geometry
394,geovelo_1447997511_13209,1447997511,None,13209,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,RUGUEUX,VELO DE ROUTE,NaT,5.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 3, 0, 0, 0, 203, 47, 131, 49, ..."
395,geovelo_11827999_13209,11827999,None,13209,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,RUGUEUX,VTC,NaT,5.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 18, 0, 0, 0, 32, 40, 183, 237,..."
396,geovelo_302219120_13209,302219120,None,13209,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,RUGUEUX,VTC,NaT,5.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 3, 0, 0, 0, 45, 100, 175, 148,..."
397,geovelo_34966336_13211,34966336,None,13211,BANDE CYCLABLE,None,UNIDIRECTIONNEL,NaN,CHAUSSEE,EN SERVICE,...,LISSE,None,NaT,NaN,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 2, 0, 0, 0, 72, 185, 69, 170, ..."
398,geovelo_828876033_13212,828876033,None,13212,PISTE CYCLABLE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,None,None,NaT,5.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 2, 0, 0, 0, 242, 112, 76, 96, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215596,geovelo_187069054_13216,187069054,None,13216,AUCUN,EN AGGLOMERATION,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,LISSE,None,NaT,30.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 19, 0, 0, 0, 100, 200, 86, 60,..."
215597,geovelo_259945150_13216,259945150,L1:V65,13216,AUCUN,EN AGGLOMERATION,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,LISSE,None,NaT,30.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 16, 0, 0, 0, 200, 200, 163, 10..."
215598,geovelo_81130528_13216,81130528,L1:V65,13216,AUCUN,EN AGGLOMERATION,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,LISSE,None,NaT,30.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 6, 0, 0, 0, 252, 231, 196, 121..."
215599,geovelo_154400529_13216,154400529,L1:V65,13216,AUCUN,None,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,LISSE,None,NaT,NaN,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"[1, 2, 0, 0, 0, 6, 0, 0, 0, 244, 217, 166, 29,..."


In [39]:

import pandas as pd
import geopandas as gpd
from shapely import wkb

df_pandas = df_amenagement_cyclable
df_pandas['geometry'] = df_pandas['geometry'].apply(lambda x: wkb.loads(bytes(x)) if x else None)


In [40]:
gdf = gpd.GeoDataFrame(df_pandas, geometry="geometry")

In [41]:

    # 2. On crée le GeoDataFrame directement à partir du DF existant
gdf = gpd.GeoDataFrame(df_pandas, geometry='geometry', crs="EPSG:4326")

    # 3. Calcul des kilomètres
    # On projette vers le système métrique (EPSG:2154 pour la France)
    # .length donne des mètres, on divise par 1000 pour les km
gdf_proj = gdf.to_crs(epsg=2154)
gdf["distance_km"] = gdf_proj.length / 1000

In [25]:
gdf

,id_local,id_osm,num_iti,code_com_d,ame_d,regime_d,sens_d,largeur_d,local_d,statut_d,...,access_ame,date_maj,trafic_vit,lumiere,d_service,source,project_c,ref_geo,geometry,distance_km
0,geovelo_1426115274_2B033,1426115274,None,2B033,AUCUN,EN AGGLOMERATION,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,None,NaT,50.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"LINESTRING (9.43758 42.67392, 9.43761 42.674, ...",0.025664
1,geovelo_621800068_2B007,621800068,None,2B007,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,VTC,NaT,5.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"LINESTRING (8.9074 42.27899, 8.90731 42.27907)",0.011541
2,geovelo_363780647_66148,363780647,,66148,AUTRE,None,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,None,NaT,NaN,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"LINESTRING (3.09948 42.49793, 3.09938 42.4978)",0.017470
3,geovelo_187713892_66093,187713892,None,66001,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,VTC,NaT,5.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"LINESTRING (2.92261 42.47938, 2.92267 42.47947...",0.066957
4,geovelo_187713892_66093,187713892,None,66093,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,VTC,NaT,5.0,NaN,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"LINESTRING (2.92298 42.47991, 2.923 42.47993, ...",9.868753
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
381851,geovelo_150112693_65192,150112693,None,65192,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,VTC,NaT,5.0,0.0,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"LINESTRING (-0.00739 42.72641, -0.00742 42.726...",0.355337
381852,geovelo_128925830_65192,128925830,None,65192,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,VTC,NaT,5.0,0.0,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"LINESTRING (-0.00826 42.72192, -0.00827 42.721...",0.670607
381853,geovelo_267690975_65192,267690975,None,65192,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,VTC,NaT,5.0,0.0,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"LINESTRING (-0.00731 42.72264, -0.00748 42.722...",0.118303
381854,geovelo_698615538_65192,698615538,None,65192,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,None,EN SERVICE,...,VTC,NaT,5.0,0.0,None,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"LINESTRING (-0.00715 42.71735, -0.00711 42.71721)",0.016018


In [42]:
df_temp_pandas = pd.DataFrame(gdf.drop(columns='geometry'))
df_temp_pandas = df_temp_pandas.dropna(subset=['code_com_d'])

In [43]:
query = """
SELECT 
        code_com_d AS code_insee,
        sum(distance_km) AS km_amenagements
    FROM df_temp_pandas
    GROUP BY code_com_d
 """

df_amenagements_par_commune = duckdb.sql(query).df()
df_amenagements_par_commune

,code_insee,km_amenagements
0,91667,4.583425
1,91549,16.926513
2,91685,1.424656
3,91434,5.569044
4,91161,19.425023
...,...,...
13408,28075,0.264442
13409,28337,0.444437
13410,28170,0.070833
13411,28249,0.383313


In [30]:
#On modifie le code insee de Paris, Lyon, Marseille pour les faire correspondre à ceux de l'INSEE
df_amenagements_par_commune.loc[df_amenagements_par_commune["code_insee"].str.startswith("75"), "code_insee"] = "75056"
df_amenagements_par_commune.loc[df_amenagements_par_commune["code_insee"].str.startswith("693"), "code_insee"] = "69123"
df_amenagements_par_commune.loc[df_amenagements_par_commune["code_insee"].str.startswith("132"), "code_insee"] = "13055"

In [44]:
query = """ 
SELECT  
    round(sum(km_amenagements),2) as km_amenagements_epci,
    df_epci.siren AS epci_code
FROM df_epci
LEFT JOIN df_amenagements_par_commune
ON df_epci.code_insee = df_amenagements_par_commune.code_insee 
GROUP BY df_epci.siren
ORDER BY km_amenagements_epci DESC
"""

df_amenagements_par_epci = duckdb.sql(query)
df_amenagements_par_epci

┌──────────────────────┬───────────┐
│ km_amenagements_epci │ epci_code │
│        double        │   int64   │
├──────────────────────┼───────────┤
│              2430.59 │ 200054781 │
│              1404.37 │ 200093201 │
│              1159.74 │ 243100518 │
│              1123.41 │ 243300316 │
│                939.2 │ 200046977 │
│               886.61 │ 244400404 │
│               825.22 │ 200054807 │
│               720.71 │ 246700488 │
│               679.51 │ 243500139 │
│               564.59 │ 200040715 │
│                   ·  │     ·     │
│                   ·  │     ·     │
│                   ·  │     ·     │
│                 NULL │ 249710062 │
│                 NULL │ 248100497 │
│                 NULL │ 241200765 │
│                 NULL │ 249740085 │
│                 NULL │ 200059871 │
│                 NULL │ 200041788 │
│                 NULL │ 247100647 │
│                 NULL │ 243200607 │
│                 NULL │ 243200409 │
│                 NULL │ 200034205 │
├

In [24]:
query = """ 
SELECT
    epci_code,
    SUM(nb_amenagements) AS total_amenagements
FROM df_amenagements_par_communes
WHERE epci_code::varchar not like '%Z'
GROUP BY epci_code
"""

df_amenagements_par_epci = duckdb.sql(query)
df_amenagements_par_epci

┌───────────┬────────────────────┐
│ epci_code │ total_amenagements │
│   int64   │       int128       │
├───────────┼────────────────────┤
│ 200069961 │                205 │
│ 200040277 │                284 │
│ 249100546 │                307 │
│ 249100595 │                 70 │
│ 249100553 │                 68 │
│ 200058014 │               3045 │
│ 200058519 │               1715 │
│ 200057974 │               1803 │
│ 200057982 │               3351 │
│ 249500430 │                 80 │
│     ·     │                  · │
│     ·     │                  · │
│     ·     │                  · │
│ 248719338 │               NULL │
│ 249740085 │               NULL │
│ 249710070 │               NULL │
│ 200060457 │               NULL │
│ 200070571 │               NULL │
│ 249740101 │               NULL │
│ 200050532 │               NULL │
│ 200027548 │               NULL │
│ 200041507 │               NULL │
│ 249740077 │               NULL │
├───────────┴────────────────────┤
│      1267 rows (20

In [35]:
df_epci[df_epci["siren"]==244400404]

,code_insee,nom,pop_tot_commune,pop_mun_commune,siren,epci_nom,epci_type,epci_modeFinancement,total_pop_tot,total_pop_mun,superficie_hectare,superficie_km2,dept_com,bassin_vie,dept_epci
16209,44109,Nantes,329809,325070,244400404,Nantes Métropole,METRO,FPU,695303,683981,6577.0,66.0,44,44109,44
16210,44162,Saint-Herblain,51282,50561,244400404,Nantes Métropole,METRO,FPU,695303,683981,2992.0,30.0,44,44109,44
16211,44143,Rezé,44055,43349,244400404,Nantes Métropole,METRO,FPU,695303,683981,1557.0,16.0,44,44109,44
16212,44190,Saint-Sébastien-sur-Loire,28996,28373,244400404,Nantes Métropole,METRO,FPU,695303,683981,1173.0,12.0,44,44109,44
16213,44114,Orvault,28949,28341,244400404,Nantes Métropole,METRO,FPU,695303,683981,2779.0,28.0,44,44109,44
16214,44215,Vertou,26487,26048,244400404,Nantes Métropole,METRO,FPU,695303,683981,3818.0,38.0,44,44109,44
16215,44047,Couëron,23808,23541,244400404,Nantes Métropole,METRO,FPU,695303,683981,4877.0,49.0,44,44109,44
16216,44026,Carquefou,21124,20535,244400404,Nantes Métropole,METRO,FPU,695303,683981,4332.0,43.0,44,44109,44
16217,44035,La Chapelle-sur-Erdre,21056,20483,244400404,Nantes Métropole,METRO,FPU,695303,683981,3339.0,33.0,44,44109,44
16218,44020,Bouguenais,20893,20590,244400404,Nantes Métropole,METRO,FPU,695303,683981,3112.0,31.0,44,44109,44


In [ ]:
query = """ 
SELECT 
    uf.*,
    ape.total_amenagements,
    ROUND(ape.total_amenagements / uf.superficie_artificialisee,2) AS amenagements_per_km2
FROM df_amenagements_par_epci ape
LEFT JOIN df_zone_urbanise_final uf
ON ape.epci_code = uf.siren
"""

df_zone_urbanise_final = duckdb.sql(query)
df_zone_urbanise_final

┌───────────┬────────────────────────────────────────────────────────────┬────────────────┬───────────────────────────┬────────────────────────────────────────┬────────────────────┬──────────────────────┐
│   siren   │                          epci_nom                          │ superficie_km2 │ superficie_artificialisee │ part_percent_superficie_artificialisee │ total_amenagements │ amenagements_per_km2 │
│   int64   │                          varchar                           │     int64      │          double           │                 double                 │       int128       │        double        │
├───────────┼────────────────────────────────────────────────────────────┼────────────────┼───────────────────────────┼────────────────────────────────────────┼────────────────────┼──────────────────────┤
│ 200000172 │ CC Faucigny-Glières                                        │            132 │                    15.745 │                                  10.41 │                209 

In [ ]:
df_zone_urbanise_final.write_csv("./data/processed/zone_urbanise_2025.csv")